# Técnicas de Otimização de Dados

## 📋 Conceitos Fundamentais

### Data File Layout
Organização e estrutura de armazenamento dos arquivos de dados subjacentes que compõem uma tabela Delta. A otimização do layout permite:
- Melhor aproveitamento dos algoritmos de **data-skipping**
- Redução do tempo de leitura em queries
- Melhoria no desempenho geral

---

## 🔧 Técnicas de Otimização

### 1. **Partitioning** (Particionamento)

**Como funciona:**
- Organiza a tabela agrupando linhas com os mesmos valores em colunas de particionamento
- Cria estrutura de diretórios físicos separados por partição

**Sintaxe:**
```sql
CREATE TABLE my_table (id INT, name STRING, year INT, month INT)
PARTITIONED BY (year)
```

**Benefícios:**
- ✅ Partition skipping - pula partições irrelevantes durante queries
- ✅ Eficiente para colunas de baixa cardinalidade

**Limitações importantes:**
- ❌ Previne compactação de arquivos entre partições
- ❌ Causa problema de **small files** (arquivos pequenos)
- ❌ Ineficiente para colunas de **alta cardinalidade**
- ❌ **Estático** - reparticionamento requer reescrita completa da tabela

📚 [Documentação Azure - Particionamento Delta Lake](https://learn.microsoft.com/pt-br/azure/databricks/tables/partitions)

---

### 2. **Z-Order Indexing** (Indexação Z-Order)

**Como funciona:**
- Agrupa dados similares em arquivos otimizados **sem criar diretórios**
- Utiliza algoritmos de data-skipping
- Efetivo para colunas de **alta cardinalidade**

**Sintaxe:**
```sql
OPTIMIZE my_table
ZORDER BY (id)
```

**Características importantes:**
- ✅ Não cria estrutura de diretórios
- ✅ Funciona bem com colunas de alta cardinalidade
- ❌ **NÃO é incremental** - cada execução reescreve os arquivos completamente

📚 [Documentação Azure - Otimização Z-Order](https://learn.microsoft.com/pt-br/azure/databricks/delta/data-skipping)

---

### 3. **Liquid Clustering** ⭐ (Recomendado)

**Como funciona:**
- Versão **melhorada** do Z-Order com mais flexibilidade e melhor desempenho
- Definição a **nível de tabela**
- Suporta **clustering incremental**

**Sintaxe:**

**Novas tabelas:**
```sql
CREATE TABLE table1 (col1 INT, col2 STRING, col3 DATE) 
CLUSTER BY (col1, col3)
```

**Tabelas existentes:**
```sql
ALTER TABLE table2 
CLUSTER BY (col1, col3)
```

**Características importantes:**
- ✅ **Incremental** - novos dados são incorporados sem reescrever tudo
- ✅ **Flexível** - pode redefinir clustering keys sem reescrever dados existentes
- ✅ Melhor desempenho que Z-Order
- ⚠️ **Incompatível** com partitioning ou ZORDER

**Escolhendo Clustering Keys:**
- Baseie-se nos padrões de query mais comuns
- Priorize colunas frequentemente usadas em filtros WHERE

📚 [Documentação Azure - Liquid Clustering](https://learn.microsoft.com/pt-br/azure/databricks/delta/clustering)

---

### 4. **Automatic Liquid Clustering** 🤖

**Como funciona:**
- Databricks **analisa automaticamente** o histórico de queries
- Escolhe as melhores clustering keys baseado no workload
- Requer **Predictive Optimization** em tabelas gerenciadas do Unity Catalog

**Sintaxe:**

**Novas tabelas:**
```sql
CREATE TABLE table1 (col1 INT, col2 STRING, col3 DATE) 
CLUSTER BY AUTO
```

**Tabelas existentes:**
```sql
ALTER TABLE table2 
CLUSTER BY AUTO
```

📚 [Documentação Azure - Otimização Preditiva](https://learn.microsoft.com/pt-br/azure/databricks/optimizations/predictive-optimization)

---

## 📊 Comparação Rápida

| Técnica | Incremental | Alta Cardinalidade | Flexibilidade | Compatibilidade |
|---------|-------------|-------------------|---------------|-----------------|
| **Partitioning** | ❌ | ❌ | Baixa | Incompatível com Clustering |
| **Z-Order** | ❌ | ✅ | Média | Incompatível com Clustering |
| **Liquid Clustering** | ✅ | ✅ | Alta | Incompatível com Partitioning/Z-Order |

---

## 💡 Boas Práticas para a Certificação

1. **Use Liquid Clustering** para novos projetos (abordagem moderna)
2. **Partitioning** é adequado apenas para colunas de baixa cardinalidade (ex: ano, região)
3. **Z-Order** é legado - prefira Liquid Clustering
4. **Nunca combine** Partitioning + Z-Order + Liquid Clustering na mesma tabela
5. Execute `OPTIMIZE` regularmente para manter o desempenho
6. Escolha clustering keys baseado em **padrões reais de query**

---

## 🔗 Links Úteis da Documentação Azure


- [Documentação Liquid Clustering Delta](https://delta.io/blog/liquid-clustering/)
- [Otimização do Delta Lake](https://learn.microsoft.com/pt-br/azure/databricks/optimizations/)
- [Gerenciamento de Arquivos](https://learn.microsoft.com/pt-br/azure/databricks/delta/file-mgmt)
- [Melhores Práticas Unity Catalog](https://learn.microsoft.com/pt-br/azure/databricks/data-governance/unity-catalog/best-practices)
- [Comando OPTIMIZE](https://learn.microsoft.com/pt-br/azure/databricks/sql/language-manual/delta-optimize)
- [Compactação de Arquivos](https://learn.microsoft.com/pt-br/azure/databricks/delta/tune-file-size)